Install Required Libraries

In [1]:
pip install langgraph langchain

Import Required Libraries

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

Define the State

In [3]:
class ExpenseState(TypedDict):
    amount_usd: float
    amount_with_tax: float
    amount_in_inr: float
    decision: str


Create Processing Functions (Graph Nodes)

In [4]:
def add_tax(state: ExpenseState):
    amount = state["amount_usd"]
    amount_with_tax = amount * 1.10
    return {"amount_with_tax": amount_with_tax}

Convert to INR Node

In [5]:
def convert_to_inr(state: ExpenseState):
    usd_amount = state["amount_with_tax"]
    inr_amount = usd_amount * 83
    return {"amount_in_inr": inr_amount}

Decision Router

In [6]:
def expense_router(state: ExpenseState):
    amount = state["amount_usd"]

    if amount <= 100:
        return "auto_approve"
    elif amount <= 1000:
        return "manager_approve"
    else:
        return "finance_approve"

Auto Approval Node

In [7]:
def auto_approve(state: ExpenseState):
    print("Expense Auto Approved")
    return {"decision": "Auto Approved"}

Manager Approval Node

In [8]:
def manager_approve(state: ExpenseState):
    print("Sent to Manager for Approval")
    return {"decision": "Manager Approved"}

Finance Approval Node

In [9]:
def finance_approve(state: ExpenseState):
    print("Sent to Finance Department for Approval")
    return {"decision": "Finance Approved"}

Final Output Node

In [10]:
def final_output(state: ExpenseState):
    print("\nFinal Result")
    print("Expense in USD:", state["amount_usd"])
    print("Amount with Tax (USD):", state["amount_with_tax"])
    print("Amount in INR:", state["amount_in_inr"])
    print("Final Decision:", state["decision"])

    return state

Build the LangGraph Workflow

In [11]:
workflow = StateGraph(ExpenseState)

Add Nodes

In [12]:
workflow.add_node("add_tax", add_tax)
workflow.add_node("convert_to_inr", convert_to_inr)
workflow.add_node("auto_approve", auto_approve)
workflow.add_node("manager_approve", manager_approve)
workflow.add_node("finance_approve", finance_approve)
workflow.add_node("final_output", final_output)

Set Entry Point

In [13]:
workflow.set_entry_point("add_tax")

Add Edges

In [14]:
workflow.add_edge("add_tax", "convert_to_inr")

Add Conditional Routing

In [15]:
workflow.add_conditional_edges(
    "convert_to_inr",
    expense_router,
    {
        "auto_approve": "auto_approve",
        "manager_approve": "manager_approve",
        "finance_approve": "finance_approve"
    }
)

Connect Approval Nodes to Final Output

In [16]:
workflow.add_edge("auto_approve", "final_output")
workflow.add_edge("manager_approve", "final_output")
workflow.add_edge("finance_approve", "final_output")

End the Graph

In [17]:
workflow.add_edge("final_output", END)

Compile the Graph

In [18]:
app = workflow.compile()

Run the Workflow

In [19]:
input_data = {
    "amount_usd": 500
}

result = app.invoke(input_data)

Sent to Manager for Approval

Final Result
Expense in USD: 500
Amount with Tax (USD): 550.0
Amount in INR: 45650.0
Final Decision: Manager Approved
